<a href="https://colab.research.google.com/github/denizstosunoglu/sifra/blob/main/SIFRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIFRA
### Shipment Intelligence for Regulatory Compliance
**RAG-powered dangerous goods compliance assistant — ADR 2025**

---
Run cells in order: Cell 1 → 2 → 3 → 4 → 5

In [54]:
!pip install -q langchain langchain-community langchain-groq faiss-cpu sentence-transformers pypdf python-dotenv gradio

In [55]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_FOLDER = "/content/drive/MyDrive/SIFRA"
PDF_PATHS = [
    os.path.join(DRIVE_FOLDER, "2412006_E_ECE_TRANS_352_Vol.I_WEB_0.pdf"),
    os.path.join(DRIVE_FOLDER, "2412010_E_ECE_TRANS_352_Vol.II_WEB.pdf"),
]
VECTORSTORE_PATH = "/content/sifra_vectorstore"
for p in PDF_PATHS:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / 1024 / 1024
        print(f"Found: {os.path.basename(p)} ({size_mb:.1f} MB)")
    else:
        print(f"NOT FOUND: {p}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: 2412006_E_ECE_TRANS_352_Vol.I_WEB_0.pdf (10.6 MB)
Found: 2412010_E_ECE_TRANS_352_Vol.II_WEB.pdf (10.5 MB)


In [56]:
# CELL 3 (REPLACED): Load vector store from Drive — no re-indexing
import shutil, os

DRIVE_VECTORSTORE = "/content/drive/MyDrive/SIFRA/vectorstore"
VECTORSTORE_PATH = "/content/sifra_vectorstore"

# Copy saved vector store from Drive to local
if os.path.exists(DRIVE_VECTORSTORE):
    if os.path.exists(VECTORSTORE_PATH):
        shutil.rmtree(VECTORSTORE_PATH)
    shutil.copytree(DRIVE_VECTORSTORE, VECTORSTORE_PATH)
    print("✅ Vector store loaded from Drive — no re-indexing needed!")
else:
    print("❌ No saved vector store found in Drive")

✅ Vector store loaded from Drive — no re-indexing needed!


In [57]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

PROMPT_TEMPLATE = """You are SIFRA, an expert dangerous goods compliance assistant.
You have access to ADR 2025 road transport regulatory documents.
Use ONLY the context below. Do not invent rules.
If context is insufficient, say so clearly.

Context:
{context}

Query: {question}

Respond in this format:
VERDICT: [COMPLIANT / NON-COMPLIANT / MISSING DATA]
UN NUMBER: [e.g. UN1090]
HAZARD CLASS: [e.g. Class 3 Flammable Liquid]
PACKING GROUP: [I / II / III]
TRANSPORT MODE: Road (ADR 2025)
REQUIRED DOCUMENTS:
- [document 1]
- [document 2]
NOTES: [conditions, exemptions, limitations]"""

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(VECTORSTORE_PATH, embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, api_key=GROQ_API_KEY)

def query_sifra(chemical, mode):
    question = f"Is {chemical} compliant for transport by {mode}? What is its UN number, hazard class, packing group, and required documents?"
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    filled_prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = llm.invoke(filled_prompt)
    sources = list({d.metadata.get("regulation", "Unknown") for d in docs})
    return response.content, f"Sources: {', '.join(sources)} — {len(docs)} chunks"

print("SIFRA ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SIFRA ready!


In [58]:
# CELL 6: Deterministic Layer — ADR Table A reference data
ADR_TABLE_A = {
    "UN1090": {"name": "Acetone", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1170": {"name": "Ethanol", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1202": {"name": "Diesel fuel / Gas oil / Heating oil", "class": "3", "pg": "III", "labels": "3", "ltd_qty": "5 L", "tunnel": "D/E"},
    "UN1203": {"name": "Petrol / Gasoline / Motor spirit", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1263": {"name": "Paint / Paint related material", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "5 L", "tunnel": "D/E"},
    "UN1993": {"name": "Flammable liquid, n.o.s.", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1219": {"name": "Isopropanol / Isopropyl alcohol", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1230": {"name": "Methanol", "class": "3", "pg": "II", "labels": "3+6.1", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1267": {"name": "Petroleum crude oil", "class": "3", "pg": "I", "labels": "3", "ltd_qty": "500 mL", "tunnel": "C/D/E"},
    "UN1294": {"name": "Toluene", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1300": {"name": "White spirit / Turpentine substitute", "class": "3", "pg": "III", "labels": "3", "ltd_qty": "5 L", "tunnel": "D/E"},
    "UN1863": {"name": "Fuel, aviation, turbine engine", "class": "3", "pg": "III", "labels": "3", "ltd_qty": "5 L", "tunnel": "D/E"},
    "UN1114": {"name": "Benzene", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1145": {"name": "Cyclohexane", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1173": {"name": "Ethyl acetate", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1208": {"name": "Hexanes", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1210": {"name": "Printing ink, flammable", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "5 L", "tunnel": "D/E"},
    "UN1648": {"name": "Acetonitrile", "class": "3", "pg": "II", "labels": "3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1760": {"name": "Corrosive liquid, n.o.s.", "class": "8", "pg": "II", "labels": "8", "ltd_qty": "1 L", "tunnel": "E"},
    "UN1789": {"name": "Hydrochloric acid", "class": "8", "pg": "II/III", "labels": "8", "ltd_qty": "1 L", "tunnel": "E"},
    "UN1805": {"name": "Phosphoric acid solution", "class": "8", "pg": "III", "labels": "8", "ltd_qty": "5 L", "tunnel": "E"},
    "UN1824": {"name": "Sodium hydroxide solution", "class": "8", "pg": "II/III", "labels": "8", "ltd_qty": "1 L", "tunnel": "E"},
    "UN1830": {"name": "Sulphuric acid", "class": "8", "pg": "II", "labels": "8", "ltd_qty": "1 L", "tunnel": "E"},
    "UN2031": {"name": "Nitric acid", "class": "8", "pg": "II", "labels": "8", "ltd_qty": "1 L", "tunnel": "E"},
    "UN2789": {"name": "Acetic acid, glacial", "class": "8", "pg": "II", "labels": "8+3", "ltd_qty": "1 L", "tunnel": "D/E"},
    "UN1005": {"name": "Ammonia, anhydrous", "class": "2.3", "pg": "-", "labels": "2.3+8", "ltd_qty": "0", "tunnel": "C/D"},
    "UN1017": {"name": "Chlorine", "class": "2.3", "pg": "-", "labels": "2.3+8", "ltd_qty": "0", "tunnel": "C/D"},
    "UN1049": {"name": "Hydrogen, compressed", "class": "2.1", "pg": "-", "labels": "2.1", "ltd_qty": "0", "tunnel": "B/D"},
    "UN1072": {"name": "Oxygen, compressed", "class": "2.2", "pg": "-", "labels": "2.2+5.1", "ltd_qty": "0", "tunnel": "E"},
    "UN1075": {"name": "Petroleum gases, liquefied / LPG", "class": "2.1", "pg": "-", "labels": "2.1", "ltd_qty": "0", "tunnel": "B/D"},
    "UN1978": {"name": "Propane", "class": "2.1", "pg": "-", "labels": "2.1", "ltd_qty": "0", "tunnel": "B/D"},
    "UN1011": {"name": "Butane", "class": "2.1", "pg": "-", "labels": "2.1", "ltd_qty": "0", "tunnel": "B/D"},
    "UN1950": {"name": "Aerosols", "class": "2", "pg": "-", "labels": "varies", "ltd_qty": "1 L", "tunnel": "D"},
    "UN3082": {"name": "Environmentally hazardous substance, liquid, n.o.s.", "class": "9", "pg": "III", "labels": "9", "ltd_qty": "5 L", "tunnel": "E"},
    "UN3480": {"name": "Lithium ion batteries", "class": "9", "pg": "-", "labels": "9A", "ltd_qty": "0", "tunnel": "E"},
    "UN3481": {"name": "Lithium ion batteries in/with equipment", "class": "9", "pg": "-", "labels": "9A", "ltd_qty": "0", "tunnel": "E"},
    "UN3090": {"name": "Lithium metal batteries", "class": "9", "pg": "-", "labels": "9A", "ltd_qty": "0", "tunnel": "E"},
    "UN1593": {"name": "Dichloromethane / Methylene chloride", "class": "6.1", "pg": "III", "labels": "6.1", "ltd_qty": "5 L", "tunnel": "E"},
    "UN2209": {"name": "Formaldehyde solution", "class": "8", "pg": "III", "labels": "8", "ltd_qty": "5 L", "tunnel": "E"},
    "UN1888": {"name": "Chloroform", "class": "6.1", "pg": "III", "labels": "6.1", "ltd_qty": "5 L", "tunnel": "E"},
}

def build_name_index():
    idx = {}
    for un, data in ADR_TABLE_A.items():
        for part in data["name"].lower().replace(" / ", "|").replace("/", "|").split("|"):
            key = part.strip()
            if key:
                idx[key] = un
    return idx

NAME_INDEX = build_name_index()

def lookup_chemical(query):
    q = query.strip().lower()
    if q.upper().replace(" ", "").startswith("UN"):
        un_key = "UN" + "".join(filter(str.isdigit, q))
        if un_key in ADR_TABLE_A:
            return un_key, ADR_TABLE_A[un_key]
    if q in NAME_INDEX:
        un = NAME_INDEX[q]
        return un, ADR_TABLE_A[un]
    for name_key, un in NAME_INDEX.items():
        if q in name_key or name_key in q:
            return un, ADR_TABLE_A[un]
    return None, None

print(f"✅ Deterministic layer ready: {len(ADR_TABLE_A)} chemicals in ADR Table A")

✅ Deterministic layer ready: 40 chemicals in ADR Table A


In [59]:
# CELL 7: 3-Layer compliance engine (Deterministic + RAG + LLM)

def get_compliance(chemical, mode="Road (ADR 2025)"):
    # LAYER 1: Deterministic lookup in ADR Table A
    un, table_data = lookup_chemical(chemical)

    # LAYER 2: RAG retrieval for transport conditions & documents
    if un:
        search_query = f"{table_data['name']} {un} class {table_data['class']} transport documents packaging requirements written instructions"
    else:
        search_query = f"{chemical} UN number hazard class packing group transport documents Table A"
    docs = retriever.invoke(search_query)
    context = "\n\n".join([d.page_content for d in docs])

    # LAYER 3: LLM synthesis
    if un:
        verified_facts = (
            f"VERIFIED FACTS from ADR Table A (authoritative):\n"
            f"- UN Number: {un}\n"
            f"- Proper Shipping Name: {table_data['name']}\n"
            f"- Hazard Class: {table_data['class']}\n"
            f"- Packing Group: {table_data['pg']}\n"
            f"- Hazard Labels: {table_data['labels']}\n"
            f"- Limited Quantity: {table_data['ltd_qty']}\n"
            f"- Tunnel Restriction Code: {table_data['tunnel']}\n"
        )
    else:
        verified_facts = "NOTE: This chemical was NOT found in the ADR Table A reference set. Rely on retrieved context and flag uncertainty."

    prompt = f"""You are SIFRA, a dangerous goods compliance assistant for road transport under ADR 2025.

{verified_facts}

Retrieved ADR context (for transport conditions and documents):
{context}

Query: Is {chemical} compliant for transport by {mode}? What documents are required?

Using the VERIFIED FACTS above as the authoritative source for classification, and the retrieved context for transport conditions, produce this report:

VERDICT: [COMPLIANT / NON-COMPLIANT / SPECIAL CONDITIONS APPLY / NOT IN DATABASE]
UN NUMBER: [from verified facts]
PROPER SHIPPING NAME: [from verified facts]
HAZARD CLASS: [from verified facts]
PACKING GROUP: [from verified facts]
HAZARD LABELS: [from verified facts]
LIMITED QUANTITY THRESHOLD: [from verified facts]
TUNNEL CODE: [from verified facts]
REQUIRED DOCUMENTS:
- [list the required transport documents based on ADR context]
NOTES: [key transport conditions, exemptions, or warnings]"""

    response = llm.invoke(prompt)

    # Build verification badge
    if un:
        badge = f"✅ VERIFIED against ADR Table A — {un}"
    else:
        badge = "⚠️ NOT FOUND in Table A reference set — LLM estimate only"

    sources = list({d.metadata.get("regulation", "Unknown") for d in docs})
    src_text = f"{badge}  |  RAG sources: {', '.join(sources)} ({len(docs)} chunks)"

    return response.content, src_text

# Quick test
answer, src = get_compliance("Acetone")
print(answer)
print()
print(src)

VERDICT: COMPLIANT
UN NUMBER: UN1090
PROPER SHIPPING NAME: Acetone
HAZARD CLASS: 3
PACKING GROUP: II
HAZARD LABELS: 3
LIMITED QUANTITY THRESHOLD: 1 L
TUNNEL CODE: D/E

REQUIRED DOCUMENTS:
- Transport document with the proper shipping name and UN number
- ADR certificate of compliance for the packaging
- Proof of quality assurance programme for the packaging manufacturer
- Proof of testing and inspection of the packaging
- If the packaging is not leak-tight or puncture-resistant, a sealed liner or bag must be used

NOTES:
- The packaging must be adequately ventilated to prevent the creation of dangerous atmospheres and the build-up of pressure.
- The transport document must be drafted in an official language of the forwarding country and also in English, French, or German, unless agreements between the countries concerned provide otherwise.
- No special conditions or exemptions apply to the transport of Acetone under ADR 2025.

✅ VERIFIED against ADR Table A — UN1090  |  RAG sources: AD

In [60]:
# CELL 9: Segregation layer — can two substances travel together?

SEGREGATION_MATRIX = {
    ("1", "2.1"): "FORBIDDEN", ("1", "3"): "FORBIDDEN", ("1", "5.1"): "FORBIDDEN",
    ("2.1", "2.1"): "OK", ("2.1", "2.3"): "CAUTION", ("2.1", "3"): "CAUTION",
    ("2.1", "5.1"): "FORBIDDEN",
    ("2.3", "6.1"): "OK",
    ("3", "3"): "OK", ("3", "5.1"): "FORBIDDEN", ("3", "5.2"): "FORBIDDEN",
    ("3", "6.1"): "CAUTION", ("3", "8"): "CAUTION",
    ("4.1", "5.1"): "FORBIDDEN", ("4.2", "5.1"): "FORBIDDEN", ("4.3", "8"): "FORBIDDEN",
    ("5.1", "5.1"): "OK", ("5.1", "6.1"): "CAUTION", ("5.1", "8"): "CAUTION",
    ("6.1", "6.1"): "OK", ("6.1", "8"): "OK",
    ("8", "8"): "CAUTION", ("9", "9"): "OK",
}

ACIDS = {"UN1789", "UN1830", "UN2031", "UN1805", "UN2789"}
BASES = {"UN1824"}

def get_segregation(c1, c2):
    if (c1, c2) in SEGREGATION_MATRIX:
        return SEGREGATION_MATRIX[(c1, c2)]
    if (c2, c1) in SEGREGATION_MATRIX:
        return SEGREGATION_MATRIX[(c2, c1)]
    if c1 == c2:
        return "OK"
    return "CAUTION"

def check_segregation_pair(chem1, chem2):
    un1, d1 = lookup_chemical(chem1)
    un2, d2 = lookup_chemical(chem2)

    if not un1:
        return f"⚠️ '{chem1}' not found in ADR Table A reference set.", ""
    if not un2:
        return f"⚠️ '{chem2}' not found in ADR Table A reference set.", ""

    c1, c2 = d1["class"], d2["class"]
    status = get_segregation(c1, c2)

    # Special: acid + base
    if (un1 in ACIDS and un2 in BASES) or (un1 in BASES and un2 in ACIDS):
        status = "FORBIDDEN"

    icons = {"OK": "🟢", "CAUTION": "🟡", "FORBIDDEN": "🔴"}
    reasons = {
        "OK": "Generally compatible for mixed loading under ADR, subject to standard packaging requirements.",
        "CAUTION": "May be transported together but require segregation: separate packaging, spacing, or securing to prevent interaction if leakage occurs.",
        "FORBIDDEN": "Dangerous reaction risk — must NOT be loaded together in the same transport unit under ADR mixed loading rules.",
    }
    if (un1 in ACIDS and un2 in BASES) or (un1 in BASES and un2 in ACIDS):
        reason_text = "Acid and base can react violently (heat/gas release). Must not be loaded together."
    else:
        reason_text = reasons[status]

    report = f"""{icons[status]} SEGREGATION VERDICT: {status}

Substance 1: {d1['name']} ({un1}) — Class {c1}, PG {d1['pg']}
Substance 2: {d2['name']} ({un2}) — Class {c2}, PG {d2['pg']}

Assessment: {reason_text}"""

    src = f"✅ Both verified against ADR Table A | Segregation per ADR 7.5.2 mixed loading rules"
    return report, src

# Test
report, src = check_segregation_pair("Sulphuric acid", "Sodium hydroxide solution")
print(report)
print()
print(src)

🔴 SEGREGATION VERDICT: FORBIDDEN

Substance 1: Sulphuric acid (UN1830) — Class 8, PG II
Substance 2: Sodium hydroxide solution (UN1824) — Class 8, PG II/III

Assessment: Acid and base can react violently (heat/gas release). Must not be loaded together.

✅ Both verified against ADR Table A | Segregation per ADR 7.5.2 mixed loading rules


In [61]:
# CELL 5 (UPDATED): SIFRA Gradio UI — 2 tabs

import gradio as gr

def ui_compliance(chemical, mode):
    if not chemical.strip():
        return "Please enter a chemical name or UN number.", ""
    try:
        return get_compliance(chemical, mode)
    except Exception as e:
        return f"Error: {str(e)}", ""

def ui_segregation(chem1, chem2):
    if not chem1.strip() or not chem2.strip():
        return "Please enter both substances.", ""
    try:
        return check_segregation_pair(chem1, chem2)
    except Exception as e:
        return f"Error: {str(e)}", ""

compliance_examples = [
    ["Acetone", "Road (ADR 2025)"],
    ["Lithium ion batteries", "Road (ADR 2025)"],
    ["Chlorine", "Road (ADR 2025)"],
    ["UN1830", "Road (ADR 2025)"],
]

segregation_examples = [
    ["Sulphuric acid", "Sodium hydroxide solution"],
    ["Acetone", "Ethanol"],
    ["Acetone", "Sulphuric acid"],
    ["Propane", "Oxygen, compressed"],
]

with gr.Blocks(title="SIFRA", theme=gr.themes.Base(primary_hue="blue")) as demo:
    gr.Markdown("# 🚛 SIFRA")
    gr.Markdown("### Shipment Intelligence for Regulatory Compliance")
    gr.Markdown("Dangerous goods compliance & segregation assistant — powered by ADR 2025")

    with gr.Tab("📋 Compliance Check"):
        gr.Markdown("Check if a single substance is compliant for road transport.")
        with gr.Row():
            with gr.Column(scale=1):
                chem = gr.Textbox(label="Chemical Name or UN Number", placeholder="e.g. Acetone or UN1090")
                mode = gr.Dropdown(label="Transport Mode", choices=["Road (ADR 2025)"], value="Road (ADR 2025)")
                btn1 = gr.Button("Check Compliance", variant="primary")
                gr.Examples(examples=compliance_examples, inputs=[chem, mode])
            with gr.Column(scale=2):
                out1 = gr.Textbox(label="Compliance Report", lines=18, interactive=False)
                src1 = gr.Textbox(label="Verification", lines=1, interactive=False)
        btn1.click(ui_compliance, [chem, mode], [out1, src1])
        chem.submit(ui_compliance, [chem, mode], [out1, src1])

    with gr.Tab("🔀 Segregation Check"):
        gr.Markdown("Check if two substances can be transported together in the same unit.")
        with gr.Row():
            with gr.Column(scale=1):
                s1 = gr.Textbox(label="Substance 1", placeholder="e.g. Sulphuric acid")
                s2 = gr.Textbox(label="Substance 2", placeholder="e.g. Sodium hydroxide solution")
                btn2 = gr.Button("Check Segregation", variant="primary")
                gr.Examples(examples=segregation_examples, inputs=[s1, s2])
            with gr.Column(scale=2):
                out2 = gr.Textbox(label="Segregation Report", lines=12, interactive=False)
                src2 = gr.Textbox(label="Verification", lines=1, interactive=False)
        btn2.click(ui_segregation, [s1, s2], [out2, src2])

    gr.Markdown("---")
    gr.Markdown("⚠️ **Disclaimer:** SIFRA is an AI assistant for educational purposes. Always verify with official ADR 2025 documentation before shipping dangerous goods.")
    gr.Markdown("*ECS Data Science & AI Program — Capstone 2026 · Deniz Tosunoglu*")

demo.launch(share=True, debug=True)

/tmp/ipykernel_2256/1813061887.py:35: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="SIFRA", theme=gr.themes.Base(primary_hue="blue")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://480f5c13e5d40b9584.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://480f5c13e5d40b9584.gradio.live
